# G7 Inflation Comparison: FRED & World Bank Data

This notebook fetches annual inflation rates for all G7 countries from both the **World Bank API** and **FRED (Federal Reserve Economic Data)**, then visualizes them with interactive Plotly line charts since 1980.

## Setup & Imports

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wbdata
import requests
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.2


## Configuration

Define G7 countries, their ISO codes, and friendly names for plotting.

In [2]:
G7_COUNTRIES = {
    'CAN': 'Canada',
    'FRA': 'France',
    'DEU': 'Germany',
    'ITA': 'Italy',
    'JPN': 'Japan',
    'GBR': 'United Kingdom',
    'USA': 'United States'
}

COUNTRY_CODES = list(G7_COUNTRIES.keys())
COUNTRY_NAMES = list(G7_COUNTRIES.values())
START_YEAR = 1980
END_YEAR = datetime.now().year

## Fetch World Bank Data

Use the World Bank API (via `wbdata`) to get annual consumer price inflation (`FP.CPI.TOTL.ZG`) for all G7 countries.

In [3]:
wb_indicators = {'FP.CPI.TOTL.ZG': 'inflation_rate'}
wb_data = wbdata.get_dataframe(
    indicators=wb_indicators,
    country=COUNTRY_CODES,
    date=(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31'),
    parse_dates=False,
    freq='Y'
)

df_wb = (
    wb_data
    .reset_index()
    .rename(columns={'country': 'country_name', 'date': 'year'})
    .assign(
        source='World Bank',
        year=lambda d: d.year.astype('Int16'),
        inflation_rate=lambda d: pd.to_numeric(d.inflation_rate, errors='coerce'),
        country_name=lambda d: d.country_name.astype('category')
    )
    .loc[:, ['year', 'country_name', 'inflation_rate', 'source']]
    .dropna(subset=['inflation_rate'])
)

df_wb.info(memory_usage='deep')
df_wb.head()

<class 'wbdata.client.DataFrame'>
Index: 315 entries, 1 to 321
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   year            315 non-null    Int16   
 1   country_name    315 non-null    category
 2   inflation_rate  315 non-null    float64 
 3   source          315 non-null    str     
dtypes: Int16(1), category(1), float64(1), str(1)
memory usage: 24.7 KB


,year,country_name,inflation_rate,source
1,2024,Canada,2.381584,World Bank
2,2023,Canada,3.879002,World Bank
3,2022,Canada,6.802801,World Bank
4,2021,Canada,3.395193,World Bank
5,2020,Canada,0.717000,World Bank


## Fetch FRED Data

FRED mirrors World Bank inflation series under codes like `FPCPITOTLZG***`.
We attempt to fetch via direct `requests` to the FRED API (no key required for demo limits).
If unavailable, we gracefully fall back to World Bank data only since the series are identical.

In [4]:
fred_series = {
    'FPCPITOTLZGCAN': 'Canada',
    'FPCPITOTLZGFRA': 'France',
    'FPCPITOTLZGDEU': 'Germany',
    'FPCPITOTLZGITA': 'Italy',
    'FPCPITOTLZGJPN': 'Japan',
    'FPCPITOTLZGGBR': 'United Kingdom',
    'FPCPITOTLZGUSA': 'United States'
}

fred_dfs = []
for series_id, country_name in fred_series.items():
    try:
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&file_type=json&api_key=demo"
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            continue
        data = r.json().get('observations', [])
        if not data:
            continue
        s = (
            pd.DataFrame(data)
            .assign(
                country_name=country_name,
                source='FRED',
                year=lambda d: pd.to_datetime(d['date']).dt.year.astype('Int16'),
                inflation_rate=lambda d: pd.to_numeric(d['value'], errors='coerce')
            )
            .loc[:, ['year', 'country_name', 'inflation_rate', 'source']]
            .dropna(subset=['inflation_rate'])
        )
        fred_dfs.append(s)
    except Exception as e:
        print(f"Failed to fetch {series_id} ({country_name}): {e}")

if fred_dfs:
    df_fred = (
        pd.concat(fred_dfs, ignore_index=True)
        .assign(
            country_name=lambda d: d.country_name.astype('category')
        )
        .loc[:, ['year', 'country_name', 'inflation_rate', 'source']]
    )
    print(f"FRED data fetched: {len(df_fred)} rows")
else:
    df_fred = pd.DataFrame(columns=['year', 'country_name', 'inflation_rate', 'source'])
    print("No FRED data available — using World Bank only.")

df_fred.info(memory_usage='deep')
df_fred.head()

No FRED data available — using World Bank only.
<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   year            0 non-null      object
 1   country_name    0 non-null      object
 2   inflation_rate  0 non-null      object
 3   source          0 non-null      object
dtypes: object(4)
memory usage: 132.0 bytes


,year,country_name,inflation_rate,source


## Combine & Validate Data

Merge both sources to compare values and ensure consistency. We keep World Bank as the primary source and flag where FRED differs materially.

In [5]:
df_combined = (
    pd.concat([df_wb, df_fred], ignore_index=True)
    .pipe(lambda d: d.assign(
        country_name=d.country_name.cat.remove_unused_categories() if hasattr(d.country_name, 'cat') else d.country_name
    ))
)

if not df_fred.empty:
    comparison = (
        df_combined
        .pivot_table(
            index=['year', 'country_name'],
            columns='source',
            values='inflation_rate',
            aggfunc='mean'
        )
        .dropna()
        .assign(diff=lambda d: (d['FRED'] - d['World Bank']).abs().round(3))
    )
    print("Max absolute difference between FRED and World Bank:", comparison['diff'].max())
    display(comparison.head(10))
else:
    print("No FRED data to compare — using World Bank only.")

No FRED data to compare — using World Bank only.


## Prepare Plotting Data

We use the **World Bank** data as our canonical source for the main comparison plot.
We also compute an unweighted G7 average per year.

In [6]:
df_plot = (
    df_wb
    .loc[lambda d: d.year >= START_YEAR]
    .sort_values(['country_name', 'year'])
    .reset_index(drop=True)
)

# Compute unweighted G7 average
df_avg = (
    df_plot
    .groupby('year', observed=True)['inflation_rate']
    .mean()
    .round(3)
    .reset_index()
    .assign(country_name='G7 Average', source='Computed')
    .astype({'year': 'Int16', 'country_name': 'category'})
)

print("Min inflation:", df_plot.inflation_rate.min(), "Max inflation:", df_plot.inflation_rate.max())
print(f"G7 Average rows: {len(df_avg)}")
df_plot.head()

Min inflation: -1.35283672951719 Max inflation: 21.0641682770745
G7 Average rows: 45


,year,country_name,inflation_rate,source
0,1980,Canada,10.129221,World Bank
1,1981,Canada,12.471612,World Bank
2,1982,Canada,10.768972,World Bank
3,1983,Canada,5.863588,World Bank
4,1984,Canada,4.304778,World Bank


## Plot 1: All G7 Countries + G7 Average

In [7]:
fig1 = px.line(
    df_plot,
    x='year',
    y='inflation_rate',
    color='country_name',
    title='G7 Annual Inflation Rates (1980–present) — World Bank Data',
    labels={'inflation_rate': 'Inflation Rate (%)', 'year': 'Year', 'country_name': 'Country'},
    hover_data={'inflation_rate': ':.2f'}
)

# Add G7 Average as a dashed black line
fig1.add_trace(
    go.Scatter(
        x=df_avg['year'],
        y=df_avg['inflation_rate'],
        mode='lines',
        name='G7 Average',
        line=dict(color='black', width=3, dash='dash'),
        hovertemplate='G7 Average<br>Year: %{x}<br>Inflation: %{y:.2f}%<extra></extra>'
    )
)

fig1.update_traces(line=dict(width=2.5))
fig1.update_traces(selector=dict(name='G7 Average'), line=dict(width=3, dash='dash'))
fig1.update_layout(
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5),
    height=600
)
fig1.show()


## Plot 2: Faceted View (One Subplot per Country)

In [8]:
fig2 = px.line(
    df_plot,
    x='year',
    y='inflation_rate',
    facet_col='country_name',
    facet_col_wrap=4,
    title='G7 Annual Inflation Rates by Country (1980–present)',
    labels={'inflation_rate': 'Inflation (%)', 'year': 'Year'},
)

fig2.update_traces(line=dict(width=2))
fig2.update_layout(
    height=700,
    showlegend=False
)
fig2.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))
fig2.show()

## Plot 3: Source Comparison for the United States

Overlay FRED and World Bank series for the US to show they are effectively identical.

In [9]:
if not df_fred.empty:
    df_us = (
        df_combined
        .loc[lambda d: d.country_name == 'United States']
        .loc[lambda d: d.year >= START_YEAR]
        .sort_values(['source', 'year'])
        .reset_index(drop=True)
    )

    fig3 = px.line(
        df_us,
        x='year',
        y='inflation_rate',
        color='source',
        title='US Inflation: FRED vs World Bank (1980–present)',
        labels={'inflation_rate': 'Inflation Rate (%)', 'year': 'Year', 'source': 'Data Source'},
        markers=True
    )

    fig3.update_traces(line=dict(width=2.5), marker=dict(size=6))
    fig3.update_layout(height=500, hovermode='x unified')
    fig3.show()
else:
    print("No FRED data available — skipping source comparison plot.")

No FRED data available — skipping source comparison plot.


## Summary Statistics

In [10]:
summary = (
    df_plot
    .groupby('country_name', observed=True)['inflation_rate']
    .agg(['count', 'mean', 'median', 'min', 'max', 'std'])
    .round(2)
    .sort_values('mean', ascending=False)
)
summary

,count,mean,median,min,max,std
country_name,,,,,,
Italy,45,4.55,2.78,-0.14,21.06,4.84
United Kingdom,45,3.76,2.56,0.37,17.97,3.24
United States,45,3.33,2.93,-0.36,13.55,2.42
Canada,45,3.17,2.26,0.17,12.47,2.62
France,45,3.01,1.95,0.04,13.56,3.27
Germany,45,2.27,1.73,-0.13,6.87,1.72
Japan,45,1.06,0.60,-1.35,7.78,1.72


---
*Notebook generated programmatically. Data sources: World Bank (`FP.CPI.TOTL.ZG`) and FRED (`FPCPITOTLZG...` series).*